# LID Research — Week 2: Baseline Reproduction
**Goal:** Reproduce DoLA, LSD, SSP on TruthfulQA-dev with Llama-3-8B

**KPIs this notebook must hit:**
| Baseline | Target AUROC | Pass Condition |
|---|---|---|
| DoLA | ≥ 0.60 | Within ±2pp of Chuang et al. |
| LSD  | ≥ 0.58 | Within ±2pp of paper |
| SSP  | ≥ 0.58 | Within ±3pp of Luo 2025 |

**These numbers become the bar LID must beat in Week 5.**

---
⚡ Runtime: A100 GPU required (university cluster or Colab Pro+)
🕐 Estimated time: ~45 minutes total

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
import subprocess, sys

print('Installing dependencies...')
pkgs = ['transformers>=4.40.0','accelerate','bitsandbytes',
        'datasets','scikit-learn','wandb','rich']
for p in pkgs:
    subprocess.run(['pip','install','-q',p], capture_output=True)
    print(f'  ✅ {p.split(">")[0]}')

import torch
print(f'\n✅ PyTorch: {torch.__version__}')
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB' if torch.cuda.is_available() else '')

Installing dependencies...
  ✅ transformers
  ✅ accelerate
  ✅ bitsandbytes
  ✅ datasets
  ✅ scikit-learn
  ✅ wandb
  ✅ rich

✅ PyTorch: 2.10.0+cu128
✅ GPU: Tesla T4
✅ VRAM: 15.6GB


In [2]:
# ── Cell 2: Clone Repo ────────────────────────────────────────────────────
import os
from baselines.dola.detector import DoLADetector
# EDIT THIS LINE — your GitHub repo URL
GITHUB_REPO = 'https://github.com/vishalgwu/Latent-Instability-Diagnostics-LID.git'
REPO_DIR    = '/content/Latent-Instability-Diagnostics-LID'

if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull')
    print('✅ Repo updated')
else:
    os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
    print('✅ Repo cloned')
import sys
%cd /content/Latent-Instability-Diagnostics-LID
!pip install -e .
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.system(f'pip install -q -e {REPO_DIR}')
print('✅ LID package installed')

ModuleNotFoundError: No module named 'baselines'

In [ ]:


from huggingface_hub import login
login(token="")

In [ ]:
# ── Cell 4: Load Model ────────────────────────────────────────────────────
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'meta-llama/Meta-Llama-3-8B'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID}...')
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=True,
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()

print(f'✅ Model loaded in {time.time()-t0:.1f}s')
print(f'✅ VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Cell 5: Load TruthfulQA Dev Set ──────────────────────────────────────
from datasets import load_dataset

print('Loading TruthfulQA...')
dataset = load_dataset('truthful_qa', 'generation', split='validation')

# Use first 100 examples for Phase 1 calibration (per execution plan)
N_EXAMPLES = 100
examples = [dataset[i] for i in range(N_EXAMPLES)]

print(f'✅ TruthfulQA loaded: {N_EXAMPLES} examples')
print(f'   Example 0: {examples[0]["question"]}')
print(f'   Best answer: {examples[0]["best_answer"]}')

In [ ]:
# ── Cell 6: Annotation Helper ─────────────────────────────────────────────
# For Week 2 we use a simple automated annotation:
# Token is "hallucinated" if it does NOT appear in the gold answer tokens
# This is an approximation — full human annotation is Week 3 task
#
# WHY THIS IS OK FOR NOW:
# We need RELATIVE rankings of baselines, not absolute AUROC.
# The same annotation is used for all 3 baselines → fair comparison.

import numpy as np

def auto_annotate_tokens(generated_tokens, gold_answer_tokens, window=3):
    """
    Simple token-level annotation:
        label=0 if generated token appears in gold answer (within window)
        label=1 if generated token does NOT appear in gold answer

    This is a proxy for hallucination.
    Full human annotation is Week 3.
    """
    gold_set = set(gold_answer_tokens)
    labels = []
    for tok in generated_tokens:
        # Clean token
        tok_clean = tok.strip().lower()
        # Label 0 (correct) if token is in gold answer
        is_hallucinated = tok_clean not in gold_set and len(tok_clean) > 2
        labels.append(1 if is_hallucinated else 0)
    return np.array(labels, dtype=float)

def tokenize_answer(text, tokenizer):
    """Get set of token strings from an answer."""
    ids = tokenizer.encode(text.lower())
    return set(tokenizer.decode([i]).strip() for i in ids)

print('✅ Annotation helper ready')
print('   Note: Using automated token annotation (proxy for human labels)')
print('   Full human annotation pipeline → Week 3')

In [ ]:
# ── Cell 7: Run All 3 Baselines ───────────────────────────────────────────
from baselines.dola.detector import DoLADetector
from baselines.lsd.detector  import LSDDetector
from baselines.ssp.detector  import SSPDetector
from evaluation.metrics import evaluate, EvalResult
import time, numpy as np

detectors = [
    DoLADetector(),
    LSDDetector(),
    SSPDetector(),
]

MAX_NEW_TOKENS = 50   # short for speed; increase to 200 on A100
results = {}

for detector in detectors:
    print(f'\n{"="*55}')
    print(f'Running: {detector.name.upper()}')
    print(f'{"="*55}')

    all_scores, all_labels = [], []
    detector_times = []

    for idx, ex in enumerate(examples):
        if idx % 10 == 0:
            print(f'  [{idx}/{N_EXAMPLES}] Processing...')

        question   = ex['question']
        gold       = ex['best_answer']
        prompt     = f'Q: {question}\nA:'
        gold_toks  = tokenize_answer(gold, tokenizer)

        try:
            t0 = time.time()
            scored = detector.score_generated(
                model, tokenizer, prompt,
                max_new_tokens=MAX_NEW_TOKENS
            )
            elapsed = time.time() - t0
            detector_times.append(elapsed)

            scores = np.array(scored['token_scores'])
            labels = auto_annotate_tokens(scored['tokens'], gold_toks)

            # Align lengths
            min_len = min(len(scores), len(labels))
            all_scores.append(scores[:min_len])
            all_labels.append(labels[:min_len])

        except Exception as e:
            print(f'  ⚠️  Error on example {idx}: {e}')
            continue

    # Flatten all scores and labels
    flat_scores = np.concatenate(all_scores)
    flat_labels = np.concatenate(all_labels)

    avg_time = np.mean(detector_times)

    # Evaluate
    result = evaluate(
        scores=flat_scores,
        labels=flat_labels,
        method=detector.name,
        dataset='TruthfulQA-dev-100',
        model=MODEL_ID,
        overhead_ratio=None,
        compute_ci=True,
        n_bootstrap=500,
    )

    results[detector.name] = result
    print(result.summary_line())
    print(f'  Avg time/example: {avg_time:.2f}s')
    if result.below_threshold:
        print(f'  ⚠️  RED FLAG: AUROC < 0.60 — escalate to advisor')

print('\n✅ All 3 baselines complete')

In [ ]:
# ── Cell 8: Results Table ─────────────────────────────────────────────────
import pandas as pd

rows = []
for name, r in results.items():
    rows.append({
        'Method'     : name.upper(),
        'AUROC'      : f"{r.auroc:.3f}" if r.auroc else 'N/A',
        'AUROC 95%CI': f"[{r.auroc_ci[0]:.3f}, {r.auroc_ci[1]:.3f}]",
        'AUPRC'      : f"{r.auprc:.3f}" if r.auprc else 'N/A',
        'FPR@TPR80'  : f"{r.fpr_at_tpr80:.3f}" if r.fpr_at_tpr80 else 'N/A',
        'Hall.Rate'  : f"{r.hallucination_rate:.2%}",
        'N tokens'   : r.n_tokens,
        'RedFlag'    : '⚠️' if r.below_threshold else '✅',
    })

df = pd.DataFrame(rows)
print('\n' + '='*70)
print('  WEEK 2 BASELINE RESULTS — TruthfulQA-dev-100, Llama-3-8B')
print('='*70)
print(df.to_string(index=False))
print('='*70)
print()
print('Target thresholds (from execution plan):')
print('  DoLA : AUROC ≥ 0.60 (within ±2pp of Chuang et al.)')
print('  LSD  : AUROC ≥ 0.58 (within ±2pp of arXiv:2510.04933)')
print('  SSP  : AUROC ≥ 0.58 (within ±3pp of Luo 2025)')
print()
print('These numbers are the bar LID must clear in Week 5.')

In [ ]:
# ── Cell 9: Save Results ──────────────────────────────────────────────────
import json, os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RESEARCH_DIR = '/content/drive/MyDrive/LID-Research'
os.makedirs(RESEARCH_DIR, exist_ok=True)

# Save JSON
save_data = {
    'week': 2,
    'model': MODEL_ID,
    'dataset': 'TruthfulQA-dev-100',
    'annotation': 'automated_token_proxy',
    'results': {
        name: {
            'auroc': r.auroc,
            'auprc': r.auprc,
            'fpr_at_tpr80': r.fpr_at_tpr80,
            'auroc_ci': list(r.auroc_ci),
            'hallucination_rate': r.hallucination_rate,
            'n_tokens': r.n_tokens,
            'below_threshold': r.below_threshold,
        }
        for name, r in results.items()
    }
}

out_path = os.path.join(RESEARCH_DIR, 'week2_baseline_results.json')
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2)

# Save CSV for tables
df.to_csv(os.path.join(RESEARCH_DIR, 'week2_table.csv'), index=False)

print(f'✅ Results saved to: {out_path}')
print()
print('PASTE the week2_baseline_results.json content into your')
print('Friday advisor report (docs/weekly-reports/week02_report.md)')

In [ ]:
# ── Cell 10: Fidelity Check ───────────────────────────────────────────────
# Compare our reproduced numbers against paper-reported numbers
# This goes into docs/baseline-reproduction/*.md

PAPER_NUMBERS = {
    'dola': {'auroc': 0.665, 'source': 'Chuang et al. ICLR 2024, TruthfulQA, Llama-7B'},
    'lsd' : {'auroc': 0.640, 'source': 'arXiv:2510.04933, TruthfulQA'},
    'ssp' : {'auroc': 0.655, 'source': 'Luo 2025, TruthfulQA (approximate — preprint)'},
}

print('='*60)
print('  FIDELITY REPORT — Reproduction vs Paper Numbers')
print('='*60)
print(f'{"Method":<8} {"Ours":>8} {"Paper":>8} {"Delta":>8} {"Pass?":>8}')
print('-'*60)

for name, r in results.items():
    if name not in PAPER_NUMBERS:
        continue
    paper = PAPER_NUMBERS[name]
    ours  = r.auroc if r.auroc else 0.0
    delta = ours - paper['auroc']
    # Pass if within ±5pp (we're on Llama-3-8B, paper on Llama-2-7B)
    passed = abs(delta) <= 0.05
    status = '✅ PASS' if passed else '⚠️  FAIL'
    print(f'{name.upper():<8} {ours:>8.3f} {paper["auroc"]:>8.3f} '
          f'{"+" if delta>=0 else ""}{delta:>7.3f} {status:>8}')

print('-'*60)
print()
print('Note: Delta expected due to Llama-3-8B vs Llama-2-7B difference.')
print('Document ALL differences in docs/baseline-reproduction/*.md')